In [2]:
import torch
import clip
from PIL import Image
import urllib.request
import pandas as pd
import numpy as np
import re
import requests
from io import BytesIO
import concurrent.futures
from tqdm import tqdm

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

In [4]:
df = pd.read_excel('../behaviour_simulation_test_company.xlsx')
df

,id,date,content,username,media,inferred company
0,1,2019-01-08 18:03:10,the Heathrow drone right now <hyperlink>,bbcthree,[Photo(previewUrl='https://pbs.twimg.com/media...,bbc
1,2,2018-01-29 10:51:17,A deal at £60m?\n\nThe latest on Pierre-Emeric...,BBCSport,[Photo(previewUrl='https://pbs.twimg.com/media...,bbc
2,3,2019-09-05 07:25:14,"""Dealing with Boris Johnson is like someone de...",BBCPolitics,[Video(thumbnailUrl='https://pbs.twimg.com/med...,bbc
3,4,2018-03-28 18:15:00,Doesn't unlimited food &amp; drinks all night ...,BrdgstoneArena,[Photo(previewUrl='https://pbs.twimg.com/media...,bridgestone
4,5,2019-09-11 21:15:44,Apple Watch Series 5.\n\nYou’ve never seen a w...,BestBuy,[Video(thumbnailUrl='https://pbs.twimg.com/ext...,best buy
...,...,...,...,...,...,...
9995,9996,2020-01-25 01:45:07,"""So what do all these defenses mean? What they...",HouseIntelDems,[Video(thumbnailUrl='https://pbs.twimg.com/med...,house
9996,9997,2019-06-26 12:51:27,"In this week’s #MacroMemo, we explore last wee...",rbcgamnews,[Photo(previewUrl='https://pbs.twimg.com/media...,rbc
9997,9998,2019-05-06 10:37:03,When it's bank holiday Monday and someone ment...,bbceastenders,[Gif(thumbnailUrl='https://pbs.twimg.com/tweet...,bbc
9998,9999,2019-05-19 12:00:51,England boss Gareth Southgate says he is conce...,BBCSport,[Photo(previewUrl='https://pbs.twimg.com/media...,bbc


In [5]:
def clean(input):
    input_string = input
    
    full_url_match = re.search(r"fullUrl='([^']+)'", input_string)
    thumbnail_url_match = re.search(r"thumbnailUrl='([^']+)'", input_string)

    if full_url_match:
        full_url = full_url_match.group(1)
        return full_url
    elif thumbnail_url_match:
        thumbnail_url = thumbnail_url_match.group(1)
        return thumbnail_url

temp = []
for i in df['media']:
    temp.append(clean(i))
df['media'] = temp

In [6]:
df

,id,date,content,username,media,inferred company
0,1,2019-01-08 18:03:10,the Heathrow drone right now <hyperlink>,bbcthree,https://pbs.twimg.com/media/DwaFKixW0AIjsun?fo...,bbc
1,2,2018-01-29 10:51:17,A deal at £60m?\n\nThe latest on Pierre-Emeric...,BBCSport,https://pbs.twimg.com/media/DUs_TqAXkAAUzvB?fo...,bbc
2,3,2019-09-05 07:25:14,"""Dealing with Boris Johnson is like someone de...",BBCPolitics,https://pbs.twimg.com/media/EDrwxl9UUAEDktR.jpg,bbc
3,4,2018-03-28 18:15:00,Doesn't unlimited food &amp; drinks all night ...,BrdgstoneArena,https://pbs.twimg.com/media/DZYxnDVXkAILX_H?fo...,bridgestone
4,5,2019-09-11 21:15:44,Apple Watch Series 5.\n\nYou’ve never seen a w...,BestBuy,https://pbs.twimg.com/ext_tw_video_thumb/11718...,best buy
...,...,...,...,...,...,...
9995,9996,2020-01-25 01:45:07,"""So what do all these defenses mean? What they...",HouseIntelDems,https://pbs.twimg.com/media/EPF0EOxX4AEx2-M.jpg,house
9996,9997,2019-06-26 12:51:27,"In this week’s #MacroMemo, we explore last wee...",rbcgamnews,https://pbs.twimg.com/media/D9_SkJaWkAAvORQ?fo...,rbc
9997,9998,2019-05-06 10:37:03,When it's bank holiday Monday and someone ment...,bbceastenders,https://pbs.twimg.com/tweet_video_thumb/D5ATAF...,bbc
9998,9999,2019-05-19 12:00:51,England boss Gareth Southgate says he is conce...,BBCSport,https://pbs.twimg.com/media/D67aeXyW4AAFJfQ?fo...,bbc


In [7]:
def process_image_from_url(url, results, index):
    try:
        response = requests.get(url)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))
        image = preprocess(image).unsqueeze(0).to(device)

        with torch.no_grad():
            image_features = model.encode_image(image)

        #print(f"Number of features for {url}: {len(image_features[0])}")

        results[index] = {
            'features': image_features[0].cpu().numpy().tolist(),
            'error': None
        }

    except Exception as e:
        #print(f"Error processing image at {url}: {e}")
        results[index] = {
            'features': None,
            'error': index
        }

#print("Error indices:", error_indices)
#print("Image embeddings:", image_embeddings)

In [8]:
image_urls = df['media']

with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    results = {}
    futures = [executor.submit(process_image_from_url, url, results, i) for i, url in enumerate(image_urls)]

    concurrent.futures.wait(futures)

error_indices = [index for index, result in results.items() if result['error'] is not None]
image_embeddings = [result['features'] for result in results.values() if result['features'] is not None]

print("Image processing completed.")

/home/bcca/anaconda3/envs/verb/lib/python3.9/site-packages/PIL/Image.py:975: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Image processing completed.


In [9]:
len(error_indices)

86

In [10]:
len(image_embeddings)

9914

In [11]:
dfie3 = pd.DataFrame(image_embeddings)
dfie3_err = pd.DataFrame(error_indices)
dfie3.to_csv("image_embeddings[:100000].csv")
dfie3_err.to_csv("image_embeddings_errors[:100000].csv")